In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')

print("\n--- 🚀 SAF VERİ MÜHENDİSLİĞİ: ORDINAL & FREKANS KODLAMA ---")

# 1. VERİLERİ YÜKLÜYORUZ (Orijinal IBM verisi dahil)
train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig = pd.read_csv(url)
orig = orig.rename(columns={'customerID': 'id'}).set_index('id')

y_train = train['Churn'].map({'Yes': 1, 'No': 0})
train.drop('Churn', axis=1, inplace=True)
y_orig = orig['Churn'].map({'Yes': 1, 'No': 0})
orig.drop('Churn', axis=1, inplace=True)

X_train_full = pd.concat([train, orig])
y_train_full = pd.concat([y_train, y_orig])
df_all = pd.concat([X_train_full, test], keys=['train', 'test'])

# -------------------------------------------------------------------
# YENİ VERİ İŞLEME 1: "Sıfırıncı Ay" ve TotalCharges Düzeltmesi
# -------------------------------------------------------------------
df_all['TotalCharges'] = pd.to_numeric(df_all['TotalCharges'], errors='coerce')
# Eğer müşteri 0 aylıksa (yeni geldiyse), toplam ödemesi ilk aylık ödemesi kadar olacaktır
df_all.loc[df_all['tenure'] == 0, 'TotalCharges'] = df_all['MonthlyCharges']
df_all['TotalCharges'].fillna(0, inplace=True) # Geri kalan nadir boşluklar için

# -------------------------------------------------------------------
# YENİ VERİ İŞLEME 2: ORDINAL ENCODING (Hiyerarşi Belirleme)
# -------------------------------------------------------------------
# Sözleşme ve İnternet için mantıksal büyüklük sırası
df_all['Contract'] = df_all['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
df_all['InternetService'] = df_all['InternetService'].map({'No': 0, 'DSL': 1, 'Fiber optic': 2})

# Ek hizmetlerin tümü için: İnternet yoksa 0, hizmet yoksa 1, hizmet varsa 2
ek_hizmetler = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in ek_hizmetler:
    df_all[col] = df_all[col].map({'No internet service': 0, 'No': 1, 'Yes': 2})

# -------------------------------------------------------------------
# YENİ VERİ İŞLEME 3: FREKANS (COUNT) KODLAMASI
# -------------------------------------------------------------------
# PaymentMethod ve MultipleLines gibi hiyerarşisi olmayanları veri setindeki bulunma sıklıklarıyla değiştiriyoruz
freq_cols = ['PaymentMethod', 'MultipleLines', 'gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in freq_cols:
    freq_map = df_all[col].value_counts(normalize=True).to_dict()
    df_all[f'{col}_Freq'] = df_all[col].map(freq_map)

df_all.drop(columns=freq_cols, inplace=True) # Orijinallerini siliyoruz, frekansları kaldı

# -------------------------------------------------------------------
# BİLDİĞİMİZ SİHİRLİ ÖZELLİKLER (Bunları ordinal formata göre güncelledik)
# -------------------------------------------------------------------
df_all['TotalCharges_Diff'] = df_all['TotalCharges'] - (df_all['MonthlyCharges'] * df_all['tenure'])
# Artık ek hizmetler "2" (Yes) ise saydırıyoruz
df_all['Total_Extra_Services'] = (df_all[ek_hizmetler] == 2).sum(axis=1)
# Contract_Months hesabı değişti, çünkü Contract artık 0, 1, 2
contract_months_map = {0: 1, 1: 12, 2: 24}
df_all['Contract_Months'] = df_all['Contract'].map(contract_months_map)
df_all['Tenure_Contract_Ratio'] = df_all['tenure'] / df_all['Contract_Months']

# Veriyi ayırıyoruz (Dikkat: Hiç One-Hot Encoding yapmadık, sadece sayısal!)
X_train_final = df_all.xs('train')
X_test_final = df_all.xs('test')

print("Tüm veriler sayısal formata çevrildi! Ağaçların en sevdiği format.")
print(f"Eğitim Seti Boyutu: {X_train_final.shape}")

# --- TEK MODEL: XGBOOST EĞİTİMİ (5-Fold) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_final))
test_preds = np.zeros(len(X_test_final))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx], y_train_full.iloc[valid_idx]
    
    # One-Hot yapmadığımız için ağaçlar çok daha derin ilişkiler kurabilecek
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[valid_idx] = valid_preds
    test_preds += model.predict_proba(X_test_final)[:, 1] / skf.n_splits
    
    print(f"Deep Feature Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc = roc_auc_score(y_train_full, oof_preds)
print(f"\n🏆 Derin Veri Mühendisliği OOF Skoru: {cv_auc:.4f}")

# Gönderim Dosyası
submission = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds})
submission.to_csv("../submissions/submission_deep_features.csv", index=False)
print("✅ Tek Model / Saf Veri gönderim dosyası kaydedildi!")


--- 🚀 SAF VERİ MÜHENDİSLİĞİ: ORDINAL & FREKANS KODLAMA ---
Tüm veriler sayısal formata çevrildi! Ağaçların en sevdiği format.
Eğitim Seti Boyutu: (601237, 23)
Deep Feature Fold 1 AUC: 0.9158
Deep Feature Fold 2 AUC: 0.9164
Deep Feature Fold 3 AUC: 0.9140
Deep Feature Fold 4 AUC: 0.9152
Deep Feature Fold 5 AUC: 0.9162

🏆 Derin Veri Mühendisliği OOF Skoru: 0.9155
✅ Tek Model / Saf Veri gönderim dosyası kaydedildi!
